In [4]:
## section here is to download technical analysis data from Peihan/Norman.

import yfinance as yf
import pandas as pd
import ta
from google.cloud import bigquery
from datetime import datetime
import json
import os

####Note that the stock_config.json must be in the same directory is your py file

# Load stock config
with open("stock_config.json") as f:
    stock_config = json.load(f)

# Load BigQuery config
with open("bq_config.json") as f:
    bq_config = json.load(f)

##extract ticker values
period = stock_config["period"]
tickers = stock_config["ticker"]

# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id_ta"]
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"


##os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = bq_config["key"]  # service account private key 


# === SET UP BIGQUERY CLIENT ===
client = bigquery.Client(project=project_id)
table_ref = f"{project_id}.{dataset_id}.{table_id}"

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    schema=[
        bigquery.SchemaField("date", "DATE"),
        bigquery.SchemaField("ticker_symbol", "STRING"),
        bigquery.SchemaField("rsi_14", "FLOAT"),
        bigquery.SchemaField("sma_50", "FLOAT"),
        bigquery.SchemaField("ema_50", "FLOAT"),
    ],
)

# === PROCESS EACH TICKER ===
for ticker in tickers:
    try:
        print(f"⏱️ {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Downloading and processing: {ticker}")

        df = yf.download(ticker, period=period, auto_adjust=True)
        df.reset_index(inplace=True)

        close_prices = df['Close']
        if isinstance(close_prices, pd.DataFrame) or len(close_prices.shape) > 1:
            close_prices = close_prices.squeeze()

        # Calculate indicators
        df['rsi_14'] = ta.momentum.RSIIndicator(close=close_prices, window=14).rsi()
        df['sma_50'] = ta.trend.SMAIndicator(close=close_prices, window=50).sma_indicator()
        df['ema_50'] = ta.trend.EMAIndicator(close=close_prices, window=50).ema_indicator()

        # Add ticker column
        df['ticker_symbol'] = ticker

        # Flatten MultiIndex columns if present
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ['_'.join(map(str, col)).strip('_') for col in df.columns.values]

        # Select columns and rename for BigQuery
        df_bq = df[['Date', 'ticker_symbol', 'rsi_14', 'sma_50', 'ema_50']].copy()
        df_bq.rename(columns={'Date': 'date'}, inplace=True)

        # Remove rows with missing indicator values
        df_bq.dropna(inplace=True)

        # Upload to BigQuery
        job = client.load_table_from_dataframe(df_bq, table_ref, job_config=job_config)
        job.result()

        print(f"✅ Uploaded {len(df_bq)} rows for {ticker} to {table_ref}.")

    except Exception as e:
        print(f"❌ Error processing {ticker}: {e}")

⏱️ 2025-06-14 17:32:56 - Downloading and processing: AAPL


[*********************100%***********************]  1 of 1 completed


✅ Uploaded 1208 rows for AAPL to meta-sanctum-461903-p3.stock_analysis.techincal_indicator.
⏱️ 2025-06-14 17:33:00 - Downloading and processing: CAT


[*********************100%***********************]  1 of 1 completed


✅ Uploaded 1208 rows for CAT to meta-sanctum-461903-p3.stock_analysis.techincal_indicator.
⏱️ 2025-06-14 17:33:04 - Downloading and processing: BA


[*********************100%***********************]  1 of 1 completed


✅ Uploaded 1208 rows for BA to meta-sanctum-461903-p3.stock_analysis.techincal_indicator.


In [5]:
### section here is to download yfinance jacob's code this is usable. Includes Peihan's orginal code.
import yfinance as yf
import pandas as pd
from google.cloud import bigquery
import json
import os



### Importanting Service account json 
### note that your service account Json needs to be in the same folder as the py file you are using or you need to specify the file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"


####Note that the stock_config.json must be in the same directory is your py file

# Load stock config
with open("stock_config.json") as f:
    stock_config = json.load(f)

# Load BigQuery config
with open("bq_config.json") as f:
    bq_config = json.load(f)


##extract ticker values
ticker_symbols = stock_config["ticker"] 
period = stock_config["period"]



# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id"]





# Collect data for each ticker
all_data = []

for symbol in ticker_symbols:
    df = yf.Ticker(symbol).history(period=period)
    df["ticker_symbol"] = symbol  # Add a column to identify the ticker
    all_data.append(df)

# Combine all into one DataFrame
combined_df = pd.concat(all_data)

# Optional: Reset index if needed
combined_df = combined_df.reset_index()

combined_df = combined_df.rename(columns={
    "Open": "open_price",
    "Close": "close_price",
    "High": "high_price",
    "Low": "low_price",
    "Volume": "volume_traded",
    "Dividends":"dividend",
    "Stock Splits":"stock_splits",
    "Date":"date"
})

########## Peihan code section################################33
##df = yf.download(ticker_symbols, period=period)
##df.columns = ['{}_{}'.format(col[0], col[1]) for col in df.columns]  # flatten columns
##df = df.reset_index()

##df = (
##    pd.melt(df, id_vars='Date', var_name='Price_Ticker', value_name='Value')
##      .assign(Price_Type=lambda x: x.Price_Ticker.str.split('_').str[0],
##              Ticker=lambda x: x.Price_Ticker.str.split('_').str[1])
##      .drop(columns='Price_Ticker')
##      .pivot_table(index=['Date', 'Ticker'], columns='Price_Type', values='Value')
##      .reset_index()
##)


##### Perhan Code Section ########################################


# 3. Initialize BigQuery client
client = bigquery.Client(project=project_id)

# 4. Load data into BigQuery
job_config = bigquery.LoadJobConfig(
    ## section here is set to over ride the data. Using append complicates the code
    write_disposition="WRITE_TRUNCATE",  # or WRITE_TRUNCATE or # "Write_Append"
    autodetect=True,
    source_format=bigquery.SourceFormat.PARQUET
)

# Load DataFrame to BigQuery
job = client.load_table_from_dataframe(
    combined_df, f"{project_id}.{dataset_id}.{table_id}", job_config=job_config
)
job.result()  # Wait for completion  # or use .history(start=..., end=...


LoadJob<project=meta-sanctum-461903-p3, location=US, id=e7d66319-66c1-4002-b094-e652054adbc0>

In [5]:
### Section here is to download fred data. Use the fred_config json to configure what you wish to pull out. 

import pandas_datareader.data as web
import datetime
from dateutil.relativedelta import relativedelta
import yfinance as yf
import pandas as pd
from google.cloud import bigquery
import json
import os


### Importanting Service account json 
### note that your service account Json needs to be in the same folder as the py file you are using or you need to specify the file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"

# Function to compute start_date based on period no longer need to specify end and 
def get_date_range_from_period(period_str):
    today = datetime.date.today()
    end_date = today

    if period_str.endswith("d"):
        delta = datetime.timedelta(days=int(period_str[:-1]))
    elif period_str.endswith("mo"):
        delta = relativedelta(months=int(period_str[:-2]))
    elif period_str.endswith("y"):
        delta = relativedelta(years=int(period_str[:-1]))
    else:
        raise ValueError(f"Invalid period: {period_str}. Use formats like '1mo', '3y', '7d'.")

    start_date = end_date - delta
    return datetime.datetime.combine(start_date, datetime.time.min), datetime.datetime.combine(end_date, datetime.time.min)

# Load configuration
with open("fred_config.json") as f:
    config = json.load(f)

with open("bq_config.json") as f:
    bq_config = json.load(f)




# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id_fred"]

# Load series info
series_config = config["series"]



# Compute date range from period
period = config.get("period", "1y")  # default to 1 year if not specified
start_date, end_date = get_date_range_from_period(period)



# Download and combine data
econ_data = pd.DataFrame()

for series_code, series_name in series_config.items():
    print(f"Downloading {series_name} ({series_code})...")
    data = web.DataReader(series_code, 'fred', start_date, end_date)
    data.columns = [series_name]
    econ_data = data if econ_data.empty else econ_data.join(data, how='outer')



econ_data.index.name = "date"


# 3. Initialize BigQuery client
client = bigquery.Client(project=project_id)

# 4. Load data into BigQuery
job_config = bigquery.LoadJobConfig(
    ## section here is set to over ride the data. Using append complicates the code
    write_disposition="WRITE_TRUNCATE",  # or WRITE_TRUNCATE or # "Write_Append"
    autodetect=True,
    source_format=bigquery.SourceFormat.PARQUET
)

# Load DataFrame to BigQuery
job = client.load_table_from_dataframe(
    econ_data, f"{project_id}.{dataset_id}.{table_id}", job_config=job_config
)
job.result()  # Wait for completion  # or use .history(start=..., end=...


LoadJob<project=meta-sanctum-461903-p3, location=US, id=ba6397e9-3dc2-4a3a-b88e-2c942f974621>

In [6]:
#### this code pulls all yfinance stock info which includes dividends and stuff sector

import pandas_datareader.data as web
import datetime
from dateutil.relativedelta import relativedelta
import yfinance as yf
import pandas as pd
from google.cloud import bigquery
import json
import os



### Importanting Service account json 
### note that your service account Json needs to be in the same folder as the py file you are using or you need to specify the file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"



# Load configuration
with open("stock_config.json") as f:
    config = json.load(f)

with open("bq_config.json") as f:
    bq_config = json.load(f)


tickers = config.get("ticker", [])


# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id_stock_info"]



# List to collect info dicts
data = []

# Loop through each ticker
for symbol in tickers:
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.get_info()

        if info:
            # Extract only selected fields
            entry = {
                "ticker_symbol": symbol,
                "company_name": info.get("longName"),
                "company_description": info.get("longBusinessSummary"),
                "country":info.get("country"),
                "sector": info.get("sector"),
                "industry": info.get("industry"),
                "market_cap": info.get("marketCap"),
                "country": info.get("country"),
                "trailing_pe":info.get("trailingPE"),
                "forward_pe":info.get("forwardPE"),
                "trailing_EPS": info.get("trailingEps"),
                "forward_EPS":info.get("forwardEps"),    
                "dividendYield": info.get("dividendYield")  



            }
            data.append(entry)
        else:
            print(f"No data found for {symbol}")
    except Exception as e:
        print(f"Error retrieving {symbol}: {e}")

# Combine into one DataFrame
df_stock_info = pd.DataFrame(data)

# Combine into one DataFrame
df_stock_info = pd.DataFrame(data)

# 3. Initialize BigQuery client
client = bigquery.Client(project=project_id)

# 4. Load data into BigQuery
job_config = bigquery.LoadJobConfig(
    ## section here is set to over ride the data. Using append complicates the code
    write_disposition="WRITE_TRUNCATE",  # or WRITE_TRUNCATE or # "Write_Append"
    autodetect=True,
    source_format=bigquery.SourceFormat.PARQUET
)

# Load DataFrame to BigQuery
job = client.load_table_from_dataframe(
    df_stock_info, f"{project_id}.{dataset_id}.{table_id}", job_config=job_config
)
job.result()  # Wait for completion  # or use .history(start=..., end=...


LoadJob<project=meta-sanctum-461903-p3, location=US, id=0fab27ef-545a-4be5-95a1-35c2f399744a>

In [7]:
####this code here pulls all the data based on the stock_config.json to return all the data into an EPS Table



##### there is NO WAY TO DETERMINE the number of records to be pulled for EPS
##### Typically only 4 years



import pandas_datareader.data as web
import datetime
from dateutil.relativedelta import relativedelta
import yfinance as yf
import pandas as pd
from google.cloud import bigquery
import json
import os



### Importanting Service account json 
### note that your service account Json needs to be in the same folder as the py file you are using or you need to specify the file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"



# Load configuration
with open("stock_config.json") as f:
    config = json.load(f)

with open("bq_config.json") as f:
    bq_config = json.load(f)


tickers = config.get("ticker", [])


# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id_stock_eps"]



# List to hold individual EPS DataFrames
all_eps_data = []

for symbol in tickers:
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.income_stmt

        # Filter EPS rows
        eps_df = df.loc[["Diluted EPS", "Basic EPS"]]

        # Transpose and tidy up
        eps_df = eps_df.T.reset_index().rename(columns={"index": "date"})
        eps_df.columns.name = None
        eps_df = eps_df.rename(columns={
            "Diluted EPS": "diluted_eps",
            "Basic EPS": "basic_eps"
        })

        # Add ticker column
        eps_df["ticker_symbol"] = symbol

        # Add to list
        all_eps_data.append(eps_df)

    except Exception as e:
        print(f"Error retrieving EPS for {symbol}: {e}")

# Combine all into one DataFrame
df_all_eps = pd.concat(all_eps_data, ignore_index=True)

# 3. Initialize BigQuery client
client = bigquery.Client(project=project_id)

# 4. Load data into BigQuery
job_config = bigquery.LoadJobConfig(
    ## section here is set to over ride the data. Using append complicates the code
    write_disposition="WRITE_TRUNCATE",  # or WRITE_TRUNCATE or # "Write_Append"
    autodetect=True,
    source_format=bigquery.SourceFormat.PARQUET
)

# Load DataFrame to BigQuery
job = client.load_table_from_dataframe(
    df_all_eps, f"{project_id}.{dataset_id}.{table_id}", job_config=job_config
)
job.result()  # Wait for completion  # or use .history(start=..., end=...




LoadJob<project=meta-sanctum-461903-p3, location=US, id=8c9328f0-0c3b-460f-bbcd-fa90316712b9>

# All code below this line is for exploration only

In [ ]:
import yfinance as yf
import pandas as pd

# List to collect info dicts
data = []

# Example list of stock symbols
tickers = ["AAPL", "GOOGL", "MSFT"]  # Replace with your actual list

# Loop through each ticker
for symbol in tickers:
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.get_info()

        if info:
            # Extract only selected fields
            entry = {
                "ticker_symbol": symbol,
                "company_name": info.get("longName"),
                "company_description": info.get("longBusinessSummary"),
                "country":info.get("country"),
                "sector": info.get("sector"),
                "industry": info.get("industry"),
                "market_cap": info.get("marketCap"),
                "country": info.get("country"),
                "trailing_pe":info.get("trailingPE"),
                "forward_pe":info.get("forwardPE"),
                "trailing_EPS": info.get("trailingEps"),
                "forward_EPS":info.get("forwardEps"),    
                "dividendYield": info.get("dividendYield")  



            }
            data.append(entry)
        else:
            print(f"No data found for {symbol}")
    except Exception as e:
        print(f"Error retrieving {symbol}: {e}")

# Combine into one DataFrame
df_stock_info = pd.DataFrame(data)

In [ ]:
df_stock_info

In [ ]:
##df_stock_info.head()
df_subset = df_stock_info[['companyOfficers', 'sectorKey']]
df_subset

In [ ]:
officer_cols = [col for col in df_stock_info.columns if 'company' in col]
print(officer_cols)


In [ ]:
import yfinance as yf

ticker = yf.Ticker("AAPL")
##print(ticker.calendar)         # Earnings, Ex-dividend
print(ticker.dividends)        # Dividend events
print(ticker.splits)           # Split events


In [ ]:
import yfinance as yf
import pandas as pd

# Define list of tickers
tickers = ["AAPL", "MSFT", "GOOGL"]

# Download historical data (close prices only)
df = yf.download(tickers, period="6mo", interval="1d")

# Step 1: Stack the multi-index columns to move Ticker into rows
df_long = df.stack(level=1).reset_index()

# Step 2: Rename columns for clarity
df_long.columns.name = None  # Remove column group name if it exists
df_long = df_long.rename(columns={
    'level_1': 'Ticker',
    'Date': 'Date'
})




df_long.rename(columns={
    'Date': 'date',
    'Open': 'open_price',
    'Close': 'close_price',
    'High': 'high_price',
    'Low': 'low_price',
    'Volume': 'volume_traded'
}, inplace=True)



df_long

In [ ]:
import yfinance as yf
import pandas as pd

# List of tickers to download
tickers = ["AAPL", "MSFT", "GOOGL"]
period = "1mo"

# Collect data for each ticker
all_data = []

for symbol in tickers:
    df = yf.Ticker(symbol).history(period=period)
    df["ticker"] = symbol  # Add a column to identify the ticker
    all_data.append(df)

# Combine all into one DataFrame
combined_df = pd.concat(all_data)

# Optional: Reset index if needed
combined_df = combined_df.reset_index()

# Preview
combined_df

In [ ]:
### section here is to download yfinance jacob's code this is usable. Includes Peihan's orginal code.
import yfinance as yf
import pandas as pd
from google.cloud import bigquery
import json
import os



### Importanting Service account json 
### note that your service account Json needs to be in the same folder as the py file you are using or you need to specify the file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"


####Note that the stock_config.json must be in the same directory is your py file

# Load stock config
with open("stock_config.json") as f:
    stock_config = json.load(f)

# Load BigQuery config
with open("bq_config.json") as f:
    bq_config = json.load(f)


##extract ticker values
ticker_symbols = stock_config["ticker"] 
period = stock_config["period"]



# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id"]





# Collect data for each ticker
all_data = []

for symbol in ticker_symbols:
    df = yf.Ticker(symbol).history(period=period)
    df["ticker"] = symbol  # Add a column to identify the ticker
    all_data.append(df)

# Combine all into one DataFrame
combined_df = pd.concat(all_data)

# Optional: Reset index if needed
combined_df = combined_df.reset_index()



########## Peihan code section################################33
##df = yf.download(ticker_symbols, period=period)
##df.columns = ['{}_{}'.format(col[0], col[1]) for col in df.columns]  # flatten columns
##df = df.reset_index()

##df = (
##    pd.melt(df, id_vars='Date', var_name='Price_Ticker', value_name='Value')
##      .assign(Price_Type=lambda x: x.Price_Ticker.str.split('_').str[0],
##              Ticker=lambda x: x.Price_Ticker.str.split('_').str[1])
##      .drop(columns='Price_Ticker')
##      .pivot_table(index=['Date', 'Ticker'], columns='Price_Type', values='Value')
##      .reset_index()
##)


##### Perhan Code Section ########################################


# 3. Initialize BigQuery client
client = bigquery.Client(project=project_id)

# 4. Load data into BigQuery
job_config = bigquery.LoadJobConfig(
    ## section here is set to over ride the data. Using append complicates the code
    write_disposition="WRITE_TRUNCATE",  # or WRITE_TRUNCATE or # "Write_Append"
    autodetect=True,
    source_format=bigquery.SourceFormat.PARQUET
)

# Load DataFrame to BigQuery
job = client.load_table_from_dataframe(
    combined_df, f"{project_id}.{dataset_id}.{table_id}", job_config=job_config
)
job.result()  # Wait for completion  # or use .history(start=..., end=...
